# Data Cleaning — PA 2022 State Senate Analysis

Goal: produce one clean precinct shapefile with population, racial demographics, and two elections worth of results to feed into the ensemble notebooks

## Data Sources

| Layer | Source |
|---|---|
| 2022 State Senate Districts | [2022 Pennsylvania State Senate Districts Approved Plan](https://redistrictingdatahub.org/dataset/2022-pennsylvania-state-senate-districts-approved-plan/) |
| 2020 Census Blocks (PL 94-171) | [Pennsylvania Block PL 94-171 2020 by Table](https://redistrictingdatahub.org/dataset/pennsylvania-block-pl-94171-2020-by-table/) |
| 2020 General Election Precincts (VEST) | [VEST 2020 Pennsylvania Precinct and Election Results](https://redistrictingdatahub.org/dataset/vest-2020-pennsylvania-precinct-and-election-results/) |
| 2016 General Election Precincts (VEST) | [VEST 2016 Pennsylvania Precinct and Election Results](https://redistrictingdatahub.org/dataset/vest-2016-pennsylvania-precinct-and-election-results/) |

In [ ]:
import geopandas as gpd
import maup
import warnings
import os
from maup import smart_repair

warnings.filterwarnings('ignore')
maup.progress.enabled = True

## Step 1: Set Paths

In [ ]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DATA = os.path.join(BASE_DIR, "data", "raw")
CLEAN_DATA = os.path.join(BASE_DIR, "data", "cleaned")

print("Repo root: ", BASE_DIR)
print("Raw data at: ", RAW_DATA)
print("Clean data at: ", CLEAN_DATA)

## Step 2: Load Raw Data

In [ ]:
precincts_2020 = gpd.read_file(os.path.join(RAW_DATA, "election", "pa_gen_20_prec", "pa_gen_20_st_sldu_prec.shp"))
precincts_2016 = gpd.read_file(os.path.join(RAW_DATA, "election", "pa_vest_16", "pa_vest_16.shp"))
# P1: Total population per block — provides TOTPOP (P0010001)                                                                                                  
blocks_p1 = gpd.read_file(os.path.join(RAW_DATA, "census", "pa_pl2020_b", "pa_pl2020_p1_b.shp"))
# P3: Race by voting age population (18+) — provides WVAP (P0030003), BVAP (P0030004)                                                                          
blocks_p3 = gpd.read_file(os.path.join(RAW_DATA, "census", "pa_pl2020_b", "pa_pl2020_p3_b.shp"))
# P4: Hispanic/Latino voting age population — provides HVAP (P0040002)                                                                                         
blocks_p4 = gpd.read_file(os.path.join(RAW_DATA, "census", "pa_pl2020_b", "pa_pl2020_p4_b.shp"))
districts = gpd.read_file(os.path.join(RAW_DATA, "districts", "pa_sldu_adopted_2022", "2022 LRC-Senate-Final.shp"))

print("2020 precincts:", precincts_2020.shape, "| CRS:", precincts_2020.crs)
print("2016 precincts:", precincts_2016.shape, "| CRS:", precincts_2016.crs)
print("Blocks p1:", blocks_p1.shape, "| CRS:", blocks_p1.crs)
print("Blocks p3:", blocks_p3.shape, "| CRS:", blocks_p3.crs)
print("Blocks p4:", blocks_p4.shape, "| CRS:", blocks_p4.crs)
print("Districts:", districts.shape, "| CRS:", districts.crs)

## Step 3: Inspect Data

In [ ]:
print("Total PA population (blocks p1): ", blocks_p1["P0010001"].sum())
print("Total 2020 Biden votes: ",   precincts_2020["G20PREDBID"].sum())
print("Total 2020 Trump votes: ",   precincts_2020["G20PRERTRU"].sum())
print("Total 2016 Clinton votes: ", precincts_2016["G16PREDCLI"].sum())
print("Total 2016 Trump votes: ",   precincts_2016["G16PRERTRU"].sum())
print("Number of senate districts: ", len(districts))

## Step 4: Reproject to UTM Zone 18N (EPSG:32618)

PA sits in UTM Zone 18N — needed for `smart_repair`'s `min_rook_length` to work in meters. Also join the three block tables (p1, p3, p4) into one GeoDataFrame here

In [ ]:
precincts_utm  = precincts_2020.to_crs(epsg=32618)
precincts16_utm = precincts_2016.to_crs(epsg=32618)
districts_utm  = districts.to_crs(epsg=32618)

'Reproject blocks and merge p1, p3, p4 into one GeoDataFrame'
blocks_utm = blocks_p1.to_crs(epsg=32618)[["GEOID20", "COUNTY", "P0010001", "geometry"]].copy()
blocks_utm = blocks_utm.merge(blocks_p3[["GEOID20", "P0030001", "P0030003", "P0030004"]], on="GEOID20")
blocks_utm = blocks_utm.merge(blocks_p4[["GEOID20", "P0040002"]], on="GEOID20")

print("All layers reprojected to UTM Zone 18N (EPSG:32618)")
print("Blocks columns:", list(blocks_utm.columns))

## Step 5: Build County Boundaries from Census Blocks

Dissolve blocks by county FIPS to get county boundaries. Using blocks (not precincts) since blocks are the authoritative Census geometry and aren't affected by precinct topology issues

In [ ]:
counties_utm = blocks_utm[["COUNTY", "geometry"]].dissolve(by="COUNTY").reset_index()

print("Number of PA counties:", len(counties_utm))
counties_utm.plot(figsize=(8, 8), edgecolor="black", color="lightblue")

## Step 6: Repair Topology

`nest_within_regions` keeps precincts from crossing county lines. `min_rook_length=30` removes shared boundaries under 30m that are digitization noise rather than real borders

In [ ]:
precincts_repaired = smart_repair(precincts_utm, nest_within_regions=counties_utm, min_rook_length=30)

print("Repaired precincts pass maup.doctor?", maup.doctor(precincts_repaired))
precincts_repaired.plot(figsize=(8, 8), edgecolor="black", linewidth=0.2, color="lightgreen")

## Step 7: Assign Blocks to Precincts

This assignment is reused for population, VAP, and 2020 vote aggregation below

In [ ]:
blocks_to_precincts = maup.assign(blocks_utm.geometry, precincts_repaired.geometry)

unassigned_blocks = blocks_to_precincts.isna().sum()
print(f"Unassigned blocks: {unassigned_blocks} out of {len(blocks_utm)}")

## Step 8: Aggregate Population to Precincts

In [ ]:
precincts_repaired["TOTPOP"] = blocks_utm["P0010001"].groupby(blocks_to_precincts).sum()

print("Total population in blocks:   ", blocks_utm["P0010001"].sum())
print("Total population in precincts:", precincts_repaired["TOTPOP"].sum())

## Step 9: Aggregate Racial VAP to Precincts

BVAP, WVAP, and HVAP for short burst analysis

In [ ]:
vap_cols = [("P0030004", "BVAP"), ("P0030003", "WVAP"), ("P0040002", "HVAP")]

for col_in, col_out in vap_cols:
    precincts_repaired[col_out] = blocks_utm[col_in].groupby(blocks_to_precincts).sum()

print("BVAP total:", precincts_repaired["BVAP"].sum())
print("WVAP total:", precincts_repaired["WVAP"].sum())
print("HVAP total:", precincts_repaired["HVAP"].sum())

## Step 10: Assign Senate Districts to Precincts

Any precincts that don't fall inside a district polygon get snapped to the nearest one so nothing is left unassigned

In [ ]:
precincts_to_districts = maup.assign(precincts_repaired.geometry, districts_utm.geometry)
precincts_repaired["SENDIST"] = precincts_to_districts.map(districts_utm["DISTRICT"])

unassigned_mask = precincts_repaired["SENDIST"].isna()
n_unassigned = unassigned_mask.sum()
print(f"Unassigned precincts: {n_unassigned}")

if n_unassigned > 0:
    print("Fixing unassigned precincts by nearest district...")
    for idx in precincts_repaired[unassigned_mask].index:
        centroid = precincts_repaired.loc[idx, "geometry"].centroid
        nearest  = districts_utm.geometry.distance(centroid).idxmin()
        precincts_repaired.loc[idx, "SENDIST"] = districts_utm.loc[nearest, "DISTRICT"]
    print(f"Remaining unassigned after fix: {precincts_repaired['SENDIST'].isna().sum()}")

precincts_repaired["SENDIST"] = precincts_repaired["SENDIST"].astype("Int64")
print("Senate districts assigned:", precincts_repaired["SENDIST"].nunique())

## Step 11: Transfer 2016 Votes to 2020 Precinct Boundaries

2016 VEST precincts have different boundaries than 2020, so we disaggregate 2016 votes to census blocks using VAP weights, then reaggregate up to 2020 precincts

In [ ]:
'Assign census blocks to 2016 VEST precincts'
blocks_to_vest16 = maup.assign(blocks_utm.geometry, precincts16_utm.geometry)

print(f"Blocks unassigned to a 2016 precinct: {blocks_to_vest16.isna().sum()}")

'Weight = block VAP / total VAP of all blocks assigned to the same 2016 precinct'
weights_2016 = blocks_utm["P0030001"] / blocks_to_vest16.map(blocks_utm["P0030001"].groupby(blocks_to_vest16).sum())
weights_2016 = weights_2016.fillna(0)

'Disaggregate 2016 votes from 2016 precincts down to blocks'
prorated_2016 = maup.prorate(blocks_to_vest16, precincts16_utm[["G16PREDCLI", "G16PRERTRU"]], weights_2016)
blocks_utm[["PRES16D", "PRES16R"]] = prorated_2016.rename(columns={"G16PREDCLI": "PRES16D", "G16PRERTRU": "PRES16R"})

print("2016 Clinton — original:", precincts_2016["G16PREDCLI"].sum(), "| after prorate:", round(blocks_utm["PRES16D"].sum()))
print("2016 Trump   — original:", precincts_2016["G16PRERTRU"].sum(), "| after prorate:", round(blocks_utm["PRES16R"].sum()))

'Aggregate block-level 2016 votes up to 2020 precincts'
elec2016_cols = ["PRES16D", "PRES16R"]
precincts_repaired[elec2016_cols] = blocks_utm[elec2016_cols].groupby(blocks_to_precincts).sum()

print("\n2016 Clinton in 2020 precincts:", round(precincts_repaired["PRES16D"].sum()))
print("2016 Trump   in 2020 precincts:", round(precincts_repaired["PRES16R"].sum()))

## Step 12: Build Final Shapefile

Keeping only the columns the ensemble notebooks need and rename to clean conventions

| Column | Description |
|---|---|
| `COUNTYFP` | County FIPS code |
| `TOTPOP` | 2020 total population |
| `BVAP` | Black voting age population |
| `WVAP` | White voting age population |
| `HVAP` | Hispanic/Latino voting age population |
| `PRES20D` | 2020 Biden votes |
| `PRES20R` | 2020 Trump votes |
| `PRES16D` | 2016 Clinton votes |
| `PRES16R` | 2016 Trump votes |
| `SENDIST` | 2022 Senate district ID |
| `geometry` | Precinct polygon |

In [ ]:
final_precincts = precincts_repaired[[
    "COUNTYFP",
    "TOTPOP",
    "BVAP",
    "WVAP",
    "HVAP",
    "G20PREDBID",
    "G20PRERTRU",
    "PRES16D",
    "PRES16R",
    "SENDIST",
    "geometry"
]].copy()

final_precincts = final_precincts.rename(columns={
    "G20PREDBID": "PRES20D",
    "G20PRERTRU": "PRES20R"
})

print("Final shapefile shape:", final_precincts.shape)
print("Columns:", list(final_precincts.columns))
print("\n{}".format(final_precincts.head()))

## Step 13: Save Cleaned Shapefile

In [ ]:
output_dir = os.path.join(CLEAN_DATA, "pa_cleaned_precincts")
os.makedirs(output_dir, exist_ok=True)

final_precincts.to_file(os.path.join(output_dir, "pa_cleaned_precincts.shp"))
print("Saved cleaned shapefile to:", output_dir)